In [47]:
import os
import requests
import pandas as pd
from io import BytesIO
from dotenv import load_dotenv

load_dotenv()


class APIClient:
    """Client pour récupérer les données Vélib et météo."""

    VELIB_URL = (
        "https://opendata.paris.fr/api/explore/v2.1/"
        "catalog/datasets/velib-disponibilite-en-temps-reel/"
        "exports/csv?use_labels=true"
    )

    """METEO_URL = (
        "https://public-api.meteofrance.fr/public/DPObs/v2/"
        "station/infrahoraire-6m?id_station=01014002&format=json"
    )"""

    METEO_URL = (
        "https://public-api.meteofrance.fr/public/DPObs/v2/"
        "station/infrahoraire-6m?id_station="
    )

    STATIONS_METEO_URL = (
        "https://public-api.meteofrance.fr/public/DPObs/v2/liste-stations"
    )

    CITY_DEPARTEMENT_CODE = "75"

    def __init__(self, timeout: int = 30):
        self.timeout = timeout
        self.session = requests.Session()
        self.token = os.getenv("TOKEN")

    def get_city_stations_id(self):
        response = self.session.get(
            self.STATIONS_METEO_URL,
            headers={
                "accept": "*/*",
                "apikey": self.token,
            },
            timeout=self.timeout,
        )

        stations_df = pd.read_csv(
            BytesIO(response.content),
            encoding="utf-8-sig",
            sep=None,
            engine="python",
            dtype={"Id_station": str},
        )
        
        id_stations_df = stations_df[stations_df["Id_station"].str.startswith("75", na=False)]["Id_station"]

        return id_stations_df

    def get_velib_dataset(self):
        response = self.session.get(
            self.dataset_url,
            timeout=self.timeout,
        )

        response.raise_for_status()

        return pd.read_csv(
            BytesIO(response.content),
            encoding="utf-8-sig",
            sep=None,
            engine="python",
        )

    def get_meteo_dataset(self):
        if not self.token:
            raise ValueError(
                "TOKEN absent du fichier .env ou non chargé."
            )

        list_stations_id = self.get_city_stations_id()

        # Liste qui récupérera les DataFrames de chaque station
        dataframes = []

        for station in list_stations_id:

            url = f"{self.dataset_url}{station}&format=json"

            response = self.session.get(
                url,
                headers={
                    "accept": "*/*",
                    "apikey": self.token,
                },
                timeout=self.timeout,
            )

            response.raise_for_status()
            data = pd.json_normalize(response.json())

            dataframes.append(data)

        if dataframes:
            return pd.concat(dataframes, ignore_index=True)

        return pd.DataFrame()


    def get_dataset(self, dataset_name: str) -> pd.DataFrame:
        """Récupère les données et les convertit en DataFrame."""

        """Choisit le dataset à récupérer."""
        if dataset_name == "velib":
            self.dataset_url = self.VELIB_URL
            dataset = self.get_velib_dataset()

        elif dataset_name == "meteo":
            self.dataset_url = self.METEO_URL
            dataset = self.get_meteo_dataset()

        else:
            raise ValueError(
                "Dataset inconnu. Choisissez 'velib' ou 'meteo'."
            )

        return dataset

    def close(self) -> None:
        self.session.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

In [48]:
def fetch_dataset(dataset: str) -> pd.DataFrame:
    with APIClient() as client:
        # Récupération des données
        return client.get_dataset(dataset)


In [ ]:
"""if __name__ =="__main__":
    print(fetch_dataset("velib"))"""

      Identifiant station                          Nom station  \
0                   40001                       Hôpital Mondor   
1                   32304                   Charcot - Benfleet   
2                   14111          Cassini - Denfert-Rochereau   
3                   32017                            Basilique   
4                   11104  Charonne - Robert et Sonia Delaunay   
...                   ...                                  ...   
1514                 5202            Arbalète - Claude Bernard   
1515                 4016               Lobau - Hôtel de Ville   
1516                15056                         Place Balard   
1517                12105                       Bercy - Villot   
1518                 8004  Malesherbes - Place de la Madeleine   

     Station en fonctionnement  Capacité de la station  \
0                          OUI                      28   
1                          OUI                      28   
2                          OUI   